In [20]:
#membuat sparksession baru
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, avg, sum as spark_sum

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Tugas4_13_Siti Azimah Izzana") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [21]:
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/

#membaca langsung dari HDFS
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)
print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(10)

Jumlah baris dari HDFS: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
|ORD-3005|2026-09-09 00:00:00|             Fashion| Purwore

In [22]:
#menghitung berapa banyak nilai kosong di kolom rating
jumlah_kosong = df_dari_hdfs.filter(col("rating").isNull()).count()
print("Jumlah rating kosong:", jumlah_kosong)

#menangani missing value dengan df.na.fill()
df_dari_hdfs = df_dari_hdfs.na.fill({"rating": 0})

#vertifikasi setelah ditangani
print("Jumlah rating kosong setelah fill:", df_dari_hdfs.filter(col("rating").isNull()).count())

Jumlah rating kosong: 204
Jumlah rating kosong setelah fill: 0


In [23]:
#mnambahkan kolom total_pendapatan = unit_terjual x harga_satuan
df_dari_hdfs = df_dari_hdfs.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

#menmbahkan kolom tier_transaksi berdasarkan total_pendapatan
df_dari_hdfs = df_dari_hdfs.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df_dari_hdfs.select("order_id", "total_pendapatan", "tier_transaksi").show(10)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
|ORD-3005|          100000|         Kecil|
|ORD-3006|           40000|         Kecil|
|ORD-3007|          720000|         Besar|
|ORD-3008|          140000|         Kecil|
|ORD-3009|          900000|         Besar|
+--------+----------------+--------------+
only showing top 10 rows



In [24]:
ringkasan_kategori = df_dari_hdfs.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(col("total_pendapatan").desc())

ringkasan_kategori.show()
print("Kategori tertinggi:", ringkasan_kategori.first()["kategori"])

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+

Kategori tertinggi: Rumah Tangga


In [25]:
kota_tier_besar = df_dari_hdfs.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(col("count").desc())

kota_tier_besar.show()
print("Kota dengan transaksi Besar terbanyak:", kota_tier_besar.first()["kota"])

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+

Kota dengan transaksi Besar terbanyak: Solo


In [27]:
rata_rating = df_dari_hdfs.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(col("rata_rata_rating").desc())

rata_rating.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



In [28]:
#menyimpan DataFrame hasil bagian C ke HDFS dalam format CSV baru
df_dari_hdfs.write.mode("overwrite").csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan_september",
    header=True
)

print("Berhasil disimpan ke HDFS.")

[Stage 68:>                                                         (0 + 1) / 1]

Berhasil disimpan ke HDFS.


In [29]:
#vertifikasi hasil penyimpanan
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_olahan_september

Found 2 items
-rw-r--r--   3 Azimah supergroup          0 2026-09-12 13:49 /user/mahasiswa/tugas4/hasil_olahan_september/_SUCCESS
-rw-r--r--   3 Azimah supergroup      92296 2026-09-12 13:49 /user/mahasiswa/tugas4/hasil_olahan_september/part-00000-f0a2ff56-b698-4d85-a8c6-f7984c946555-c000.csv


In [30]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
